## Exercise - Build a Leakage-Safe Forecasting Dataset
Create a new DataFrame called `forecast_data`. The target variable should be `target_demand_1h` - electricity demand one hour in the future.

Create features that would be available before the target hour:
- current `demand`
- `hour`
- `weekday`
- `is_weekend`
- cyclical encodings for `hour`
- cyclical encodings for `weekday`
- `lag1`
- `lag24`
- `lag168`
- `roll_mean_24h`
- `roll_max_24h`
- temperature `T`

Make sure that rolling features do **not** use the current or future target value.

Remove rows with missing values after creating the features.

---

### Questions

Answer the following questions based on the created `forecast_data` DataFrame:

1. What does `target_demand_1h` represent? Why should it not be included as an input feature?
2. What is the shape of `forecast_data`?
3. What is the correlation between current `demand` and `target_demand_1h`?
4. What is the correlation between `lag24` and `target_demand_1h`?
5. What is the correlation between temperature `T` and `target_demand_1h`?
6. What does the negative correlation between temperature `T` and `target_demand_1h` suggest?


### Answers

1. `target_demand_1h` represents electricity demand one hour after the current timestamp. It should not be included as an input feature because it is the answer the model is supposed to predict. Including it would cause target leakage.
2.  `(8563, 15)`.
3. Approximately `0.968`.
4. Approximately `0.849`.
5. Approximately `-0.475`.
6. It suggests that lower temperatures are associated with higher electricity demand. This makes sense in Estonia because colder weather can increase heating-related electricity use.

In [1]:
# Imports
import datetime
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Preassemble to match existing lab materials
elering = pd.read_csv('../data/electricity-production and consumption_2022.csv',  delimiter=';',decimal=',')
elering['timestamp'] = pd.to_datetime(elering['Kuupaev (Eesti aeg)'], dayfirst=True)
elering.set_index('timestamp', inplace=True)
demand = elering[['Tarbimine']]
demand = demand.rename(columns = {'Tarbimine':'demand'})
weather = pd.read_csv('../data/weather_2022.csv',  delimiter=';', decimal='.', index_col = False)
weather['timestamp'] = pd.to_datetime(weather['Local time in Tallinn (airport)'], dayfirst=True)

In [10]:
forecast_data = demand[["demand"]].copy()

forecast_data["target_demand_1h"] = forecast_data["demand"].shift(-1)

forecast_data["hour"] = forecast_data.index.hour
forecast_data["weekday"] = forecast_data.index.dayofweek
forecast_data["is_weekend"] = (forecast_data["weekday"] >= 5).astype(int)

forecast_data["hour_sin"] = np.sin(2 * np.pi * forecast_data["hour"] / 24)
forecast_data["hour_cos"] = np.cos(2 * np.pi * forecast_data["hour"] / 24)

forecast_data["weekday_sin"] = np.sin(2 * np.pi * forecast_data["weekday"] / 7)
forecast_data["weekday_cos"] = np.cos(2 * np.pi * forecast_data["weekday"] / 7)

forecast_data["lag1"] = forecast_data["demand"].shift(1)
forecast_data["lag24"] = forecast_data["demand"].shift(24)
forecast_data["lag168"] = forecast_data["demand"].shift(168)

forecast_data["roll_mean_24h"] = forecast_data["demand"].shift(1).rolling(24).mean()
forecast_data["roll_max_24h"] = forecast_data["demand"].shift(1).rolling(24).max()

temperature = weather[["timestamp", "T"]].copy()
temperature.set_index("timestamp", inplace=True)

forecast_data = forecast_data.join(temperature)

forecast_data = forecast_data.dropna()

forecast_data.head()

,demand,target_demand_1h,hour,weekday,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos,lag1,lag24,lag168,roll_mean_24h,roll_max_24h,T
timestamp,,,,,,,,,,,,,,,
2022-01-08 00:00:00,1004.3,982.6,0,5,1,0.000000,1.000000,-0.974928,-0.222521,1039.8,955.7,899.4,1173.325000,1363.8,-8.9
2022-01-08 01:00:00,982.6,962.7,1,5,1,0.258819,0.965926,-0.974928,-0.222521,1004.3,929.1,892.1,1175.350000,1363.8,-10.2
2022-01-08 02:00:00,962.7,960.8,2,5,1,0.500000,0.866025,-0.974928,-0.222521,982.6,918.8,874.3,1177.579167,1363.8,-10.4
2022-01-08 03:00:00,960.8,955.1,3,5,1,0.707107,0.707107,-0.974928,-0.222521,962.7,906.4,860.1,1179.408333,1363.8,-10.9
2022-01-08 04:00:00,955.1,963.3,4,5,1,0.866025,0.500000,-0.974928,-0.222521,960.8,908.0,842.7,1181.675000,1363.8,-10.3


In [9]:
print("Shape:", forecast_data.shape)
print("First timestamp:", forecast_data.index.min())
print("Last timestamp:", forecast_data.index.max())
print("Missing values:", forecast_data.isna().sum().sum())

print("Correlation: demand vs target")
print(forecast_data["demand"].corr(forecast_data["target_demand_1h"]))

print("Correlation: lag24 vs target")
print(forecast_data["lag24"].corr(forecast_data["target_demand_1h"]))

print("Correlation: temperature vs target")
print(forecast_data["T"].corr(forecast_data["target_demand_1h"]))

Shape: (8563, 15)
First timestamp: 2022-01-08 00:00:00
Last timestamp: 2022-12-31 22:00:00
Missing values: 0
Correlation: demand vs target
0.9676638187965341
Correlation: lag24 vs target
0.8494074369197661
Correlation: temperature vs target
-0.47514458067440035
